In [1]:
# -*- coding: utf-8 -*-
"""1D standard FD benchmark based on Hundsdorfer--Verwer IMEX-CNLF."""

from __future__ import annotations

import os
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch

from hv_cnlf_common_verified import (
    CaseOptions,
    load_fixed_lhs,
    run_imex_cnlf_case,
    save_case_outputs,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64
BASE_PATH = os.environ.get("BENCHMARK_BASE_PATH", ".")
OUTPUT_DIR = os.environ.get(
    "BENCHMARK_OUTPUT_DIR",
    os.path.join(BASE_PATH, "verified_fdm_1d_outputs"),
)

X_MIN, X_MAX = 0.0, 1.0
T_FINAL = 0.3
LEFT_BC, RIGHT_BC = -10.0, 5.0

MU_LIST = [1.0e-02,1.0e-3, 1.0e-4]
DAE_TARGET_E2 = {1.0e-02: 4.6102e-04, 1.0e-3: 2.0294e-03, 1.0e-4: 4.5923e-03}
NUM_SAMPLES = 5000
EVAL_WARMUP = 20
EVAL_REPEAT = 200

# Ordered bracketing sequences.  The reported equivalent grid is the first
# tested pair satisfying e2 <= the FINAL DAE target.  Update DAE_TARGET_E2 from
# the final DAE table before launching these runs.
# Direct DAE-LHS extraction requires Nx-1 and Nt-1 to be multiples of 200.
CASES: Dict[float, List[Tuple[int, int]]] = {
    1.0e-2: [
        (40001, 120001),
        (44001, 132001),
        (48001, 144001),
        (50001, 150001),
    ],
    1.0e-3: [
        (2401, 16001),
        (2801, 18001),
        (3201, 20001),
    ],
    1.0e-4: [
        (10001, 40001),
        (12001, 44001),
        (14001, 48001),
        (15001, 50001),
    ],
}


def reference_filename(mu: float) -> str:
    return f"1d_U0_true_mu{mu:.0e}_201_201_Mathematica.csv"


def phi_minus(x: torch.Tensor) -> torch.Tensor:
    return -torch.sqrt(600.0 + 6.0 * x**2 - 4.0 * x**3 + 3.0 * x**4) / np.sqrt(6.0)


def phi_plus(x: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(145.0 + 6.0 * x**2 - 4.0 * x**3 + 3.0 * x**4) / np.sqrt(6.0)


def asymptotic_initial_condition(x: torch.Tensor, mu: float) -> torch.Tensor:
    """Composite asymptotic initial value from the user's previous solver."""
    h0 = torch.as_tensor(0.1, dtype=x.dtype, device=x.device)
    pm_x, pp_x = phi_minus(x), phi_plus(x)
    pm_h, pp_h = phi_minus(h0), phi_plus(h0)
    jump = pp_h - pm_h

    arg_left = torch.clamp((x - h0) * (pm_h - pp_h) / (2.0 * mu), -500.0, 500.0)
    u_left = pm_x + 0.5 * jump * (1.0 - torch.tanh(0.5 * arg_left))

    arg_right = torch.clamp((x - h0) * (pp_h - pm_h) / (2.0 * mu), -500.0, 500.0)
    u_right = pp_x - 0.5 * jump * (1.0 - torch.tanh(0.5 * arg_right))
    return torch.where(x <= h0, u_left, u_right)


def source_f(x: torch.Tensor) -> torch.Tensor:
    return x - x**2 + x**3


def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("=" * 96)
    print("1D Hundsdorfer--Verwer CD2--IMEX-CNLF benchmark")
    print(f"device={DEVICE}, dtype={DTYPE}")
    print("T_total = T_solve + T_eval; setup/error/CPU transfer/output excluded")
    print("Fixed DAE LHS set; direct grid-node extraction; no interpolation")
    print("=" * 96)

    rows = []
    for mu in MU_LIST:
        lhs = load_fixed_lhs(
            reference_path=os.path.join(BASE_PATH, reference_filename(mu)),
            coordinate_columns=("t", "x"),
            index_path=os.path.join(BASE_PATH, f"1d_LHS_sample_indices_mu{mu:.0e}.npy"),
            num_samples=NUM_SAMPLES,
            save_points_path=os.path.join(OUTPUT_DIR, f"1d_HV_LHS_points_mu{mu:.0e}.csv"),
        )

        for nx, nt in CASES[mu]:
            print("-" * 96)
            print(f"mu={mu:.0e}, Nx={nx}, Nt={nt}")
            options = CaseOptions(
                dim=1,
                mu=mu,
                grid=(nx,),
                nt=nt,
                bounds=((X_MIN, X_MAX),),
                t_final=T_FINAL,
                left_bc=LEFT_BC,
                right_bc=RIGHT_BC,
                dtype=DTYPE,
                device=DEVICE,
                eval_warmup=EVAL_WARMUP,
                eval_repeat=EVAL_REPEAT,
                finite_check_interval=max(1, (nt - 1) // 100),
                progress_every=max(1, (nt - 1) // 20),
                cfl_warning=1.0,
            )
            try:
                result = run_imex_cnlf_case(
                    options=options,
                    lhs_data=lhs,
                    source_function=source_f,
                    initial_function=asymptotic_initial_condition,
                )
                result["target_e2"] = float(DAE_TARGET_E2[mu])
                result["pass_target"] = bool(
                    float(result["e2"]) <= float(DAE_TARGET_E2[mu])
                )
                prefix = os.path.join(
                    OUTPUT_DIR, f"1d_HV_IMEX_CNLF_mu{mu:.0e}_Nx{nx}_Nt{nt}"
                )
                pred_path, runtime_path = save_case_outputs(
                    result, lhs, ("t", "x"), prefix
                )
                row = {k: v for k, v in result.items() if k != "prediction"}
                row.update({
                    "status": "success", "failure_reason": "",
                    "Nx": nx, "prediction_csv": pred_path,
                    "runtime_json": runtime_path,
                })
                print(
                    f"e2={result['e2']:.6e}, einf={result['einf']:.6e}, "
                    f"T_solve={result['T_solve']:.6f}s, "
                    f"T_eval={result['T_eval']:.6e}s, "
                    f"T_total={result['T_total']:.6f}s, "
                    "peak GPU allocated="
                    f"{result['peak_gpu_allocated_gib']:.3f} GiB"
                )
            except (RuntimeError, MemoryError) as exc:
                row = {
                    "status": "failed", "failure_reason": str(exc),
                    "mu": mu, "Nx": nx, "Nt": nt,
                    "target_e2": float(DAE_TARGET_E2[mu]),
                    "pass_target": False,
                }
                print(f"FAILED: {exc}")
            rows.append(row)
            pd.DataFrame(rows).to_csv(
                os.path.join(OUTPUT_DIR, "1d_HV_IMEX_CNLF_summary.csv"), index=False
            )
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    summary = os.path.join(OUTPUT_DIR, "1d_HV_IMEX_CNLF_summary.csv")
    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(summary, index=False)
    selected_path = os.path.join(OUTPUT_DIR, "1d_HV_IMEX_CNLF_selected.csv")
    if not summary_df.empty and "pass_target" in summary_df.columns:
        passed = summary_df[
            (summary_df["status"] == "success")
            & summary_df["pass_target"].fillna(False).astype(bool)
        ].copy()
        if not passed.empty:
            sort_columns = [c for c in ("mu", "spatial_unknowns", "Nt") if c in passed.columns]
            passed = passed.sort_values(sort_columns)
            selected = passed.groupby("mu", as_index=False).first()
            selected.to_csv(selected_path, index=False)
        else:
            pd.DataFrame(columns=summary_df.columns).to_csv(selected_path, index=False)
    print(f"Saved summary: {summary}")
    print(f"Saved selected passing cases: {selected_path}")


if __name__ == "__main__":
    main()


1D Hundsdorfer--Verwer CD2--IMEX-CNLF benchmark
device=cuda, dtype=torch.float64
T_total = T_solve + T_eval; setup/error/CPU transfer/output excluded
Fixed DAE LHS set; direct grid-node extraction; no interpolation
Loaded fixed DAE LHS set: N_test=5000, reference=1d_U0_true_mu1e-02_201_201_Mathematica.csv, indices=1d_LHS_sample_indices_mu1e-02.npy
------------------------------------------------------------------------------------------------
mu=1e-02, Nx=40001, Nt=120001
step=1/120000, t=2.500000e-06, max|u|=1.000030e+01
step=6000/120000, t=1.500000e-02, max|u|=1.000066e+01
step=12000/120000, t=3.000000e-02, max|u|=1.000113e+01
step=18000/120000, t=4.500000e-02, max|u|=1.000171e+01
step=24000/120000, t=6.000000e-02, max|u|=1.000239e+01
step=30000/120000, t=7.500000e-02, max|u|=1.000318e+01
step=36000/120000, t=9.000000e-02, max|u|=1.000406e+01
step=42000/120000, t=1.050000e-01, max|u|=1.000505e+01
step=48000/120000, t=1.200000e-01, max|u|=1.000612e+01
step=54000/120000, t=1.350000e-01